In [1]:
import pandas as pd
from pathlib import Path

# Target file path where we want to append the info
period_path = Path(r"E:\ProyectoAnalisisElectrico\DiaPromedio\Periodos\2505_2604\2505_2604_mean_period.parquet")

# Path to our geographic "source of truth"
infra_path = Path(r"E:\ProyectoAnalisisElectrico\BarrasEstaciones\infraestructura_consolidada.parquet")

print("Paths configured successfully.")

Paths configured successfully.


In [2]:
# %% [markdown]
# ### Cell 2: Data Loading and Subsetting
# Loads the parquet files and selects only the required geographic columns.

print("Loading files...")
df_period = pd.read_parquet(period_path)
df_infra = pd.read_parquet(infra_path)

print(f"Rows in the period file: {len(df_period)}\n")
print("Columns in period file:", df_period.columns.tolist())

# Select only the columns we need to merge
geo_columns = ['nombre_barra', 'tension', 'region', 'macrozona', 'confianza', 'IA']
df_infra_sub = df_infra[geo_columns]

Loading files...
Rows in the period file: 163056

Columns in period file: ['clave', 'Zona', 'Hora', 'active_calendar', 'RUT', 'rut_log', 'n_ruts', 'Razon_Social', 'razon_social_log', 'n_razones_sociales', 'Nombre_Corto', 'nombre_corto_log', 'n_nombres_cortos', 'nombre_barra', 'nombre_barra_log', 'n_nombres_barra', 'tension', 'tension_log', 'n_tensiones', 'tipo', 'period', 'medida_mean', 'medida_std', 'CMg[CLP/KWh]_mean', 'CMg[CLP/KWh]_std', 'CMg[CLP/KWh]_count', 'valorizado_CLP_mean', 'valorizado_CLP_std', 'valorizado_CLP_count', 'medida_total']


In [3]:
# %% [markdown]
# ### Cell 3: Data Cleaning (Deduplication)
# Cleans the infrastructure dataset to prevent Cartesian explosions during the merge.

# 1. Count records BEFORE cleaning
total_before = len(df_infra_sub)

# 2. Drop duplicates (using nombre_barra and tension as the key)
df_infra_geo = df_infra_sub.drop_duplicates(subset=['nombre_barra', 'tension'])

# 3. Count records AFTER and calculate how many were removed
total_after = len(df_infra_geo)
duplicates_removed = total_before - total_after

print("--- MASTER DATABASE DUPLICATE REVIEW ---")
print(f"Total initial records in infrastructure: {total_before}")
print(f"Duplicates removed: {duplicates_removed}")
print(f"Total unique records ready for merge: {total_after}\n")

--- MASTER DATABASE DUPLICATE REVIEW ---
Total initial records in infrastructure: 1420
Duplicates removed: 0
Total unique records ready for merge: 1420



In [4]:
# %% [markdown]
# ### Cell 4: Merging and Validation
# Performs the Left Join and audits the results for missing matches.

# Perform the merge (Left Join ensures we don't lose data from df_period)
df_enriched = pd.merge(
    df_period,
    df_infra_geo,
    on=['nombre_barra', 'tension'], # Using both as a key to avoid ambiguity
    how='left'
)

# Quick review to see if any bars were left without a match (total orphans)
missing_matches = df_enriched['macrozona'].isna().sum()

print("\n--- MERGE RESULT ---")
print(f"Original rows: {len(df_period)}")
print(f"Rows after merge: {len(df_enriched)}")
print(f"Bars that did not find a geographic match: {missing_matches}")

# Visualize the final result
display(df_enriched.head())


--- MERGE RESULT ---
Original rows: 163056
Rows after merge: 163056
Bars that did not find a geographic match: 0


,clave,Zona,Hora,active_calendar,RUT,rut_log,n_ruts,Razon_Social,razon_social_log,n_razones_sociales,...,CMg[CLP/KWh]_std,CMg[CLP/KWh]_count,valorizado_CLP_mean,valorizado_CLP_std,valorizado_CLP_count,medida_total,region,macrozona,confianza,IA
0,$C$439,Norte,0,111111111111,96.505.760-9,96.505.760-9,1,Colbún S.A.,Colbún S.A.,1,...,28.893765,365,-1.071117e+06,548120.866499,365,-373927.649231,Antofagasta,Norte Grande,100.0,False
1,$C$439,Norte,1,111111111111,96.505.760-9,96.505.760-9,1,Colbún S.A.,Colbún S.A.,1,...,26.857940,365,-1.043339e+06,510586.365746,365,-373927.649231,Antofagasta,Norte Grande,100.0,False
2,$C$439,Norte,2,111111111111,96.505.760-9,96.505.760-9,1,Colbún S.A.,Colbún S.A.,1,...,29.019703,365,-1.017505e+06,531921.939887,365,-373927.649231,Antofagasta,Norte Grande,100.0,False
3,$C$439,Norte,3,111111111111,96.505.760-9,96.505.760-9,1,Colbún S.A.,Colbún S.A.,1,...,33.489338,365,-1.052726e+06,557024.101790,365,-373927.649231,Antofagasta,Norte Grande,100.0,False
4,$C$439,Norte,4,111111111111,96.505.760-9,96.505.760-9,1,Colbún S.A.,Colbún S.A.,1,...,29.518448,365,-1.043940e+06,476478.617274,365,-373927.649231,Antofagasta,Norte Grande,100.0,False


In [5]:
# %% [markdown]
# ### Cell 5: Exporting
# Saves the enriched DataFrame to a new parquet file.

output_path = period_path.parent / "2505_2604_mean_period_loc.parquet"

df_enriched.to_parquet(output_path, engine="pyarrow", compression="snappy")

print(f"Enriched file saved successfully to:\n{output_path}")

Enriched file saved successfully to:
E:\ProyectoAnalisisElectrico\DiaPromedio\Periodos\2505_2604\2505_2604_mean_period_loc.parquet
